Project 1, dubbed "Salty Winter"
(Lede Program)

# Research question: We had a harsh winter. Did New York City use the highest amount of road salt to melt snow over the past winter? 

# Goal: focusing on getting and cleaning the data I need, even if visualization is not beautiful.

# Due date: June 28th, 2026

# Overall workflow: Use API to get data, then save raw CSV (archive), clean DataFrame, then analysis, then charts/visual. It's a simple project by design so keeping everything in one notebook.

Step 1. Getting data: fetch data from NYC Open Data, look at them, and save each as a CSV.
Step 2. Cleaning data: practice API, pandas, chart tools
Step 3. Visualize data: scale visualization(some sort of visual to showcase SCALE). 
        If time allows, I'll also show one reservoir location on map to show salt-related chemical increase in water.
*Document the learning process for future reference because I'll soon forget about everything.

Dataset: one single dataset (keep it simple)
Department of Sanitation (DSNY) Salt Usage
source: https://data.cityofnewyork.us/City-Government/DSNY-Salt-Usage/tavr-zknk/about_data
*note: I had oringially thought about other datasets, such as DEP Drinking Water Quality Distribution Monitoring, but scrapped that idea to avoid overthinking.

API workflow:

Build the URL
Make the request with requests.get()
Convert JSON to Python data with df
Create a DataFrame
Inspect the results

---
## how NYC Open Data's API works (what kind of API I'm dealing with)
# The website took me to formal documentation about Socrata APIs
https://dev.socrata.com/docs/queries/
# some how-to tutorials and resources:
this one explains Socrata API Basics:
https://github.com/mebauer/sodapy-tutorial-nyc-opendata/blob/main/socrata-api-basics.ipynb

# data summary:
NYC Open Data runs on a platform called Socrata (something new to me). Every dataset has its own URL that ends in '.json'. that's the API endpoint. e.g. https://data.cityofnewyork.us/resource/DATASET-ID.json

The dataset ID, unlike Pokeman API, is that  8-character code in the URL of any dataset page (e.g. 'tavr-zknk'). Can find it by querying the dataset page on NYC Open Data and clicking "API" in the top right , or just looking at the URL. 

Department of Sanitation (DSNY) Salt Usage
 https://data.cityofnewyork.us/City-Government/DSNY-Salt-Usage/tavr-zknk/about_data
 The intro page actually shows what's in the dataframe, which is helpful. I do want to somehow get the "Total Tons
 The total tons of salt DSNY dispensed onto roadways."
The plan now is: to get the total tons data for five boroughs for the LATEST winter season; then zoom in the top borough (Queens? and get historical data just for that borough)

#Poking around on the website
--found a specific documentation for this dataset
https://dev.socrata.com/foundry/data.cityofnewyork.us/tavr-zknk
--API end point
https://data.cityofnewyork.us/api/v3/views/tavr-zknk/query.json?

In [1]:
# Setting up tools, based_URL.
# Trying to get the data with API 
# Import the tools needed:

import requests
import pandas as pd

BASE_URL = "https://data.cityofnewyork.us/resource/"

print("Base URL ready.")


Base URL ready.


In [2]:
# Step 1. Getting data
# Fetching the salt data
# Dataset ID: tavr-zknk
# Build the URL manually

salt_url = BASE_URL + "tavr-zknk.json"

print("Requesting:", salt_url)

salt_response = requests.get(salt_url)

print("Status code:", salt_response.status_code)

salt_data = salt_response.json()

print("Rows returned:", len(salt_data))

df_salt = pd.DataFrame(salt_data)

print("DataFrame created!!!")

df_salt.to_csv("df_salt_raw.csv", index=False)

print("Raw data saved.")

Requesting: https://data.cityofnewyork.us/resource/tavr-zknk.json
Status code: 200
Rows returned: 224
DataFrame created!!!
Raw data saved.


In [3]:
#checking raw data
df_salt.head()

,dsny_storm,date_of_report,manhattan,bronx,brooklyn,queens,staten_island,total_tons
0,Storm 1,2016-01-19T00:00:00.000,1111,2059,2690,6625,2931,15416
1,Storm 2,2016-01-23T00:00:00.000,2919,4567,5521,9253,4264,26524
2,Storm 2,2016-01-24T00:00:00.000,6582,8167,12290,17133,5411,49583
3,Storm 2,2016-01-25T00:00:00.000,3862,4094,6217,10236,1256,25665
4,Storm 2,2016-01-26T00:00:00.000,2499,3299,5120,7685,764,19367


In [4]:
df_salt.columns

Index(['dsny_storm', 'date_of_report', 'manhattan', 'bronx', 'brooklyn',
       'queens', 'staten_island', 'total_tons'],
      dtype='str')

In [5]:
df_salt.shape

(224, 8)

In [6]:
df_salt.tail()

,dsny_storm,date_of_report,manhattan,bronx,brooklyn,queens,staten_island,total_tons
219,Storm 4,2026-01-17T00:00:00.000,2734,3253,3392,8678,1730,19787
220,Storm 5,2026-01-18T00:00:00.000,8517,5322,11710,5698,8647,39894
221,Storm 6,2026-01-25T00:00:00.000,22666,20449,34078,48369,21401,146963
222,Storm 7,2026-02-15T00:00:00.000,3210,2240,4653,5888,3341,19332
223,Storm 8/9,2026-02-22T00:00:00.000,21355,17811,36323,41657,16612,133758


Step 2. Cleaning data: 

#OK. It's a ten-year span showing salt use by each storm. I need to figure out:
How much salt was spread during the most recent winter?

I need to:

Convert the date column to a datetime.
Create a winter-season label/column.
Sum all storms within each winter.

In [7]:
df_salt['date_of_report'] = pd.to_datetime(df_salt['date_of_report'])
df_salt['year'] = df_salt['date_of_report'].dt.year
df_salt['month'] = df_salt['date_of_report'].dt.month

In [8]:
df_salt['winter_start'] = df_salt['year']
df_salt.loc[
    df_salt['month'].isin([1, 2, 3]),
    'winter_start'
] = df_salt['year'] - 1

df_salt['winter_season'] = (
    df_salt['winter_start'].astype(str)
    + "-"
    + (df_salt['winter_start'] + 1).astype(str).str[-2:]
)

In [9]:
df_salt.shape

(224, 12)

In [10]:
df_salt.to_csv("df_salt_winter.csv", index=False)

print("winter data saved.")

winter data saved.


In [11]:
#checking winter data
df_salt.tail()    

,dsny_storm,date_of_report,manhattan,bronx,brooklyn,queens,staten_island,total_tons,year,month,winter_start,winter_season
219,Storm 4,2026-01-17,2734,3253,3392,8678,1730,19787,2026,1,2025,2025-26
220,Storm 5,2026-01-18,8517,5322,11710,5698,8647,39894,2026,1,2025,2025-26
221,Storm 6,2026-01-25,22666,20449,34078,48369,21401,146963,2026,1,2025,2025-26
222,Storm 7,2026-02-15,3210,2240,4653,5888,3341,19332,2026,2,2025,2025-26
223,Storm 8/9,2026-02-22,21355,17811,36323,41657,16612,133758,2026,2,2025,2025-26


#see which season has the highest use of salt: group by 'winter_season', then sort 
*** ran the code below and found that datatype the 'total_tons' is string not a number ***
??? is there a code to see the datatype of all the columns?
# df_season_salt = df_salt.groupby('winter_season')['total_tons'].sum().reset_index().sort_values(by='total_tons', ascending=False)

In [16]:
# retry: need to convert the total_tons column into actual mathematical integers or floats before runninggroupby().
# fix the data type, recalculate the sum, and sort the results properly:
# 1. Force the total_tons column to be numeric (errors='coerce' turns bad data like text into NaN)
df_salt['total_tons'] = pd.to_numeric(df_salt['total_tons'], errors='coerce')

# 2. Fill any missing rows with 0 just in case
df_salt['total_tons'] = df_salt['total_tons'].fillna(0)

# 3. Now run your group and sum code!
df_season_salt = df_salt.groupby('winter_season')['total_tons'].sum().reset_index().sort_values(by='total_tons', ascending=False)

# 4. View the real mathematical results
df_season_salt.head(10)


,winter_season,total_tons
10,2025-26,504811
5,2020-21,453763
2,2017-18,433394
3,2018-19,368066
1,2016-17,363191
6,2021-22,337890
0,2015-16,266265
9,2024-25,234845
4,2019-20,203502
8,2023-24,195757


In [17]:
#save the df sorted by total_tons of each winter_season into a csv
df_season_salt.to_csv("df_salt_winter_sorted by total tons.csv", index=False)

print("Sorted data saved.")

Sorted data saved.


In [18]:
# next query is to find out over the most recent winter season, ie 2025-2026, which borough used the most salt. 
import pandas as pd

# 1. (lesson learned) make sure all borough columns are numeric instead of text strings
borough_cols = ['manhattan', 'bronx', 'brooklyn', 'queens', 'staten_island']
for col in borough_cols:
    df_salt[col] = pd.to_numeric(df_salt[col], errors='coerce').fillna(0)

# 2. filter for the 2025-26 season and sum the columns
totals_by_borough = df_salt[df_salt['winter_season'] == '2025-26'][borough_cols].sum()

# 3. Turn it into a clean, sorted DataFrame named df_season_borough
df_season_borough = totals_by_borough.reset_index()
df_season_borough.columns = ['borough', 'total_tons']
df_season_borough = df_season_borough.sort_values(by='total_tons', ascending=False)

# 4. View the new DataFrame
df_season_borough


,borough,total_tons
3,queens,153925
2,brooklyn,127156
0,manhattan,82901
4,staten_island,71536
1,bronx,69293


In [20]:
#wow, let's save that into yet another csv
df_season_borough.to_csv("df_season_borough.csv", index=False)

print("borough data saved.")

print(df_salt['queens'].dtype)

borough data saved.
int64


In [22]:
#let's look at the historical data for queens (not sure if will use these data or how)
# Group by season, sum the numeric Queens data, and sort chronologically
df_queens_history = df_salt.groupby('winter_season')['queens'].sum().reset_index().sort_values(by='winter_season', ascending=False)

# View the clean historical data
df_queens_history


,winter_season,queens
10,2025-26,153925
9,2024-25,77911
8,2023-24,67919
7,2022-23,31477
6,2021-22,109997
5,2020-21,179784
4,2019-20,81261
3,2018-19,118757
2,2017-18,140208
1,2016-17,113953


In [24]:
#let's save the queens data into yet another csv. Maybe useful for visuals/charts/map?
df_queens_history.to_csv("df_queens_history.csv", index=False)

print("Queens data saved.")

Queens data saved.


# Step 3. Visualize data: 
# I think I'm ready to make two charts. 
# I would like to chart with Altair, which I just learned it this morning. how should I do that? I wanted to create two charts with df_season_salt and df_season_borough, to show city-wide, total tons of last winter was the highest, and by borough, for last winter season, queens was the highest, respectively


In [34]:
#quitely install altair

%pip install -q altair vega_datasets


import altair as alt

#building city-wide chart, but keeping it them vertical with flat labels

# Build the city-wide historical chart
chart_city_wide_flat = alt.Chart(df_season_salt).mark_bar().encode(
    # 'labelAngle=0' forces the winter season text to stay perfectly horizontal
    x=alt.X('winter_season:N', title='Winter Season', axis=alt.Axis(labelAngle=0)),
    # 'total_tons:Q' plots the numeric quantity on the vertical axis
    y=alt.Y('total_tons:Q', title='Total Tons of Salt Used'),
    # Adds interactive hover tooltips for digital news layouts
    tooltip=['winter_season', 'total_tons']
).properties(
    title='NYC Total Salt Usage by Winter Season',
    width=600,
    height=400
)

# Display the final chart
chart_city_wide_flat




Note: you may need to restart the kernel to use updated packages.


alt.Chart(...)

In [30]:
#building 2nd chart: historical data for Queens

# Build the borough breakdown chart
chart_borough_flat = alt.Chart(df_season_borough).mark_bar().encode(
    # 'labelAngle=0' ensures borough names are perfectly upright and easy to read
    x=alt.X('borough:N', title='Borough', sort='-y', axis=alt.Axis(labelAngle=0)),
    # Plots total borough tonnage on the vertical axis
    y=alt.Y('total_tons:Q', title='Tons of Salt Used (2025-26)'),
    # highlights Queens in orange, mutes others in blue-gray
    color=alt.condition(
        alt.datum.borough == 'queens', 
        alt.value('#f58518'), # Queens Orange
        alt.value('#72b7b2')  # Other Boroughs Muted Blue
    ),
    # Adds interactive hover tooltips for readers
    tooltip=['borough', 'total_tons']
).properties(
    title='Salt Usage by Borough for the 2025-26 Winter Season',
    width=500,
    height=400
)

# Display the final chart
chart_borough_flat



alt.Chart(...)

Now that we know the use of salt was highest last winter season, I want to put it in scale, somehow.
# per capita idea:
If DSNY handed out the rock salt evenly to every resident at the start of the winter, every single person in Queens would have to lug home a xxx pound block of salt (kg)
# need to creat a "pounds per resident" column 

# import pandas as pd

# 1. Map the approximate current population to each borough
# Source: Recent U.S. Census Bureau estimates (2025/2026 data)
borough_populations = {
    'brooklyn': 2649000,
    'queens': 2358000,
    'manhattan': 1664000,
    'bronx': 1424000,
    'staten_island': 493000
}

# 2. Add the population data into your borough dataframe
df_season_borough['population'] = df_season_borough['borough'].map(borough_populations)

# 3. Math calculation: (Total Tons * 2000 pounds per ton) / Population
df_season_borough['pounds_per_resident'] = (df_season_borough['total_tons'] * 2000) / df_season_borough['population']

# 4. Round the results to one decimal point for cleaner reading
df_season_borough['pounds_per_resident'] = df_season_borough['pounds_per_resident'].round(1)

# View the final dataset
df_season_borough_perperson=df_season_borough.sort_values(by='pounds_per_resident', ascending=False)

df_season_borough_perperson.to_csv("df_season_borough_perperson.csv", index=False)

print("Per capita data saved.")

df_season_borough_perperson

# Not sure how I would use the per capita salt data but good to have.
# Now, time to try and scale the data. This is new to me. 
the instructor mentioned these in class:
visual stories for inspiration:
https://www.reuters.com/graphics/ENVIRONMENT-PLASTIC/0100B275155/
# Can I try scrolllhma? to show the increase use of road salt over time?
https://www.reuters.com/graphics/USA-ECONOMY/FAIR-INSURANCE/lgpdqnqamvo/

Wanted to go with this approach: mention the Empire State Building comparison (roughly 1.4 ESBs) in the story ("NYC spread half a million tons of road salt, last winter, equivalent of 1.4 Empire State Buildings' worth of salt last winter", just as the plastic story), then use the city-truck pictogram as the actual graphic (just like the house insruance story) (Note: I end up removing the Empire State Building comparison to avoid distracting readers)
# googled: how to scrollama in jupyter notebook (Note: I end up using the template from class)

#I'll need to create the images first: for three seasons for scrolling purposes, right?
#how to create the three images? (Note: I end up using illustrator to sketch the truck icon. My son helped me with the truck-drawing process)

*Reporting notes: I removed the last bit of reporting about reservoir salt level after fact checking with the Department of Environment Protection. 
What I learned most from this experience/project is, not creating new columns, making charts or using pictograms, but explaining one point clearly with one dataset and making sure the point I'm trying to make is fair when crossreferencing another dataset that I'm not farmiliar with.